In [9]:
# [CELL 1] STUDENT CONFIGURATION
from pathlib import Path

LAST_NAME = 'Magalong'
STUDENT_ID = 'TUPM-26-1134'

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())
NAME_LENGTH = len(LAST_NAME)

print('Seed:', SEED_DIGIT)
print('ID Sum:', ID_SUM)
print('Name Length:', NAME_LENGTH)

Seed: 4
ID Sum: 17
Name Length: 8


In [10]:
# [CELL 2] CREATE ACTIVITY WORKSPACE
WORK_DIR = Path('.')
DATA_DIR = WORK_DIR / 'data'
OUTPUT_DIR = WORK_DIR / 'output'
BACKUP_DIR = WORK_DIR / 'backup'

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

print('Student:', LAST_NAME)
print('Student ID:', STUDENT_ID)
print('Workspace:', WORK_DIR.resolve())
print('Data folder exists:', DATA_DIR.exists())
print('Output folder exists:', OUTPUT_DIR.exists())
print('Backup folder exists:', BACKUP_DIR.exists())

Student: Magalong
Student ID: TUPM-26-1134
Workspace: C:\Users\Paul\Documents\TUPM-BSECE\1C\AECEL1\Lab\Lab6_File_Handling_Magalong\Lab6_Additional_Exercises
Data folder exists: True
Output folder exists: True
Backup folder exists: True


In [11]:
# [CELL 3] EXERCISE 1 PARAMETERS
import csv
import json
import shutil
from collections import Counter

dataset_path = DATA_DIR / 'weblog.csv'

records_to_process = 1000 + ID_SUM
target_class = (SEED_DIGIT % 4) + 2

print('Dataset:', dataset_path)
print('Dataset exists:', dataset_path.exists())
print('Records to process:', records_to_process)
print(f'Target status class: {target_class}xx')

Dataset: data\weblog.csv
Dataset exists: True
Records to process: 1017
Target status class: 2xx


In [12]:
# [CELL 4] READ LOG LINE BY LINE AND EXTRACT FIELDS
records_processed = 0
invalid_records = 0
selected_records = []

with open(dataset_path, 'r', encoding='utf-8', newline='') as file:
    reader = csv.reader(file)
    header = next(reader)
    columns = {name.strip(): index for index, name in enumerate(header)}

    print('Header:', header)

    for row in reader:
        if records_processed >= records_to_process:
            break

        records_processed += 1

        try:
            ip = row[columns['IP']].strip()
            timestamp = row[columns['Time']].strip()
            url = row[columns['URL']].strip()
            status = row[columns['Staus']].strip()

            if not (ip and timestamp and url and status.isdigit()):
                raise ValueError('missing or invalid field')

            if int(status) // 100 == target_class:
                selected_records.append([ip, timestamp, url, status])

        except (IndexError, KeyError, ValueError):
            invalid_records += 1

print('Records processed:', records_processed)
print('Invalid or missing records:', invalid_records)
print('Records in target class:', len(selected_records))

Header: ['IP', 'Time', 'URL', 'Staus']
Records processed: 1017
Invalid or missing records: 0
Records in target class: 848


In [13]:
# [CELL 5] COUNT STATUS CODES AND TOP URLS
status_counts = Counter(record[3] for record in selected_records)

def get_resource(url):
    parts = url.split()
    return parts[1] if len(parts) >= 2 else url

url_counts = Counter(get_resource(record[2]) for record in selected_records)
top_five_urls = url_counts.most_common(5)

print('Count per status code:')
for status, count in sorted(status_counts.items()):
    print(f'  {status}: {count}')

print('Five most requested URLs:')
for rank, (url, count) in enumerate(top_five_urls, 1):
    print(f'  {rank}. {url} - {count}')

Count per status code:
  200: 840
  206: 8
Five most requested URLs:
  1. /login.php - 105
  2. /home.php - 91
  3. /css/normalize.css - 38
  4. /css/bootstrap.min.css - 37
  5. /js/vendor/jquery-1.12.0.min.js - 37


In [14]:
# [CELL 6] EXPORT FILTERED CSV
filtered_csv_path = OUTPUT_DIR / f'{LAST_NAME}_{STUDENT_ID}_weblog_filtered.csv'

with open(filtered_csv_path, 'w', encoding='utf-8', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['IP', 'Time', 'URL', 'Status'])
    writer.writerows(selected_records)

print('Created:', filtered_csv_path)
print('Exported rows:', len(selected_records))
print('File size:', filtered_csv_path.stat().st_size, 'bytes')

Created: output\Magalong_TUPM-26-1134_weblog_filtered.csv
Exported rows: 848
File size: 62787 bytes


In [15]:
# [CELL 7] WRITE JSON SUMMARY
summary_path = OUTPUT_DIR / f'{LAST_NAME}_{STUDENT_ID}_weblog_summary.json'

summary = {
    'last_name': LAST_NAME,
    'student_id': STUDENT_ID,
    'seed_digit': SEED_DIGIT,
    'id_sum': ID_SUM,
    'records_processed': records_processed,
    'invalid_records': invalid_records,
    'target_status_class': f'{target_class}xx',
    'records_in_target_class': len(selected_records),
    'count_per_status_code': dict(sorted(status_counts.items())),
    'top_five_urls': [{'url': url, 'count': count} for url, count in top_five_urls]
}

with open(summary_path, 'w', encoding='utf-8') as file:
    json.dump(summary, file, indent=4)

print('Created:', summary_path)

with open(summary_path, 'r', encoding='utf-8') as file:
    print(file.read())

Created: output\Magalong_TUPM-26-1134_weblog_summary.json
{
    "last_name": "Magalong",
    "student_id": "TUPM-26-1134",
    "seed_digit": 4,
    "id_sum": 17,
    "records_processed": 1017,
    "invalid_records": 0,
    "target_status_class": "2xx",
    "records_in_target_class": 848,
    "count_per_status_code": {
        "200": 840,
        "206": 8
    },
    "top_five_urls": [
        {
            "url": "/login.php",
            "count": 105
        },
        {
            "url": "/home.php",
            "count": 91
        },
        {
            "url": "/css/normalize.css",
            "count": 38
        },
        {
            "url": "/css/bootstrap.min.css",
            "count": 37
        },
        {
            "url": "/js/vendor/jquery-1.12.0.min.js",
            "count": 37
        }
    ]
}


In [16]:
# [CELL 8] BACKUP JSON AND VERIFY
backup_path = BACKUP_DIR / summary_path.name

shutil.copy2(summary_path, backup_path)

backup_exists = backup_path.exists()
sizes_match = backup_exists and backup_path.stat().st_size == summary_path.stat().st_size

print('Original:', summary_path)
print('Backup:', backup_path)
print('Backup exists:', backup_exists)
print('Backup size matches original:', sizes_match)
print('Backup verification:', 'VERIFIED' if sizes_match else 'CHECK')

Original: output\Magalong_TUPM-26-1134_weblog_summary.json
Backup: backup\Magalong_TUPM-26-1134_weblog_summary.json
Backup exists: True
Backup size matches original: True
Backup verification: VERIFIED


In [17]:
# [CELL 9] FINAL OUTPUT
print('ASSESSMENT DATA - WEB SERVER ACCESS LOG ANALYSIS')
print('------------------------------------------------')
print('Records processed:', records_processed)
print('Target status class:', f'{target_class}xx')
print('Records in target class:', len(selected_records))
print('Count per status code:', dict(sorted(status_counts.items())))
print('Five most requested URLs:')
for rank, (url, count) in enumerate(top_five_urls, 1):
    print(f'  {rank}. {url} ({count})')
print('Filtered CSV file:', filtered_csv_path.name)
print('JSON summary file:', summary_path.name)
print('Backup verification:', 'VERIFIED' if sizes_match else 'CHECK')

ASSESSMENT DATA - WEB SERVER ACCESS LOG ANALYSIS
------------------------------------------------
Records processed: 1017
Target status class: 2xx
Records in target class: 848
Count per status code: {'200': 840, '206': 8}
Five most requested URLs:
  1. /login.php (105)
  2. /home.php (91)
  3. /css/normalize.css (38)
  4. /css/bootstrap.min.css (37)
  5. /js/vendor/jquery-1.12.0.min.js (37)
Filtered CSV file: Magalong_TUPM-26-1134_weblog_filtered.csv
JSON summary file: Magalong_TUPM-26-1134_weblog_summary.json
Backup verification: VERIFIED
